In [1]:
import json, inspect
import urllib.request
from openai import OpenAI
from openai.types.chat import ChatCompletionMessage, ChatCompletionToolParam

from typing import Literal, TypedDict, Annotated, Any, Callable, get_type_hints, get_origin, get_args

client = OpenAI()
MODEL = "gpt-4o-mini"
API_DOMAIN = "https://nomad-movies-2.nomadcoders.workers.dev"

class CallFunction(TypedDict):
  name: str
  arguments: str

class ToolCall(TypedDict):
  id: str
  type: Literal["function"]
  function: CallFunction

class Message(TypedDict):
  role: Literal["user", "assistant", "tool"]
  content: Any
  tool_call_id: str | None
  tool_name: str | None
  tool_calls: list[ToolCall] | None


messages: list[Message] = []

In [2]:
# TOOLS
def get_popular_movies():
  """
    Retrieves a list of currently popular movies. 
    Use this tool when the user asks what movies are popular or trending right now, or wants general suggestions of well-known current movies, WITHOUT naming a specific title or ID. 
    Returns a list of movies, each including its movie ID, title, release date, short overview, and popularity/rating. 
    The movie IDs returned here can be reused as the `id` argument for get_movie_details and get_movie_credits.
    When responding to the user with data from this tool, you MUST include each movie's ID alongside its title (e.g., "Fight Club (ID: 550)"). Never omit the movie ID from the response.
  """

  with urllib.request.urlopen(urllib.request.Request(f"{API_DOMAIN}/movies", headers = { "User-Agent": "Mozilla/5.0" })) as response:
    return json.loads(response.read())

def get_movie_details(id: Annotated[str, "The unique numeric identifier of the movie (e.g., 550). Obtain it from a get_popular_movies result or directly from the user. Do not guess or invent this value."]):
  """
    Retrieves detailed information about a single movie identified by its numeric movie ID. 
    Use this tool when the user asks about a specific movie's information such as plot/overview, release date, runtime, genres, or rating, and a movie ID is available. 
    This tool does NOT return cast or crew — use get_movie_credits for that.
  """

  with urllib.request.urlopen(urllib.request.Request(f"{API_DOMAIN}/movies/{id}", headers = { "User-Agent": "Mozilla/5.0" })) as response:
    return json.loads(response.read())

def get_movie_credits(id: Annotated[str, "The unique numeric identifier of the movie (e.g., 550). Obtain it from a get_popular_movies result or directly from the user. Do not guess or invent this value."]):
  """
    Retrieves the cast and crew of a single movie identified by its numeric movie ID.
    Use this tool when the user asks who appears in, who stars in, who directed, or who worked on a specific movie. 
    Returns the cast (actors and the characters they play) and the crew (people and their jobs, e.g., director, writer). 
    This tool does NOT return general movie details like plot or release date — use get_movie_details for that.
  """
  
  with urllib.request.urlopen(urllib.request.Request(f"{API_DOMAIN}/movies/{id}/credits", headers = { "User-Agent": "Mozilla/5.0" })) as response:
    return json.loads(response.read())

def get_similar_movies(id: Annotated[str, "The unique numeric identifier of the movie (e.g., 550). Obtain it from a get_popular_movies result or directly from the user. Do not guess or invent this value."]):
  """
    Retrieves a list of movies similar to a single movie identified by its numeric movie ID.
    Use this tool when the user asks for recommendations of movies similar to, like, or in the same vein as a specific movie, and a movie ID is available.
    Returns a list of movies, each including its movie ID, title, release date, short overview, and popularity/rating.
    The movie IDs returned here can be reused as the `id` argument for get_movie_details and get_movie_credits.
    When responding to the user with data from this tool, you MUST include each movie's ID alongside its title (e.g., "Fight Club (ID: 550)"). Never omit the movie ID from the response.
  """

  with urllib.request.urlopen(urllib.request.Request(f"{API_DOMAIN}/movies/{id}/similar", headers = { "User-Agent": "Mozilla/5.0" })) as response:
    return json.loads(response.read())

FUNCTION_MAP = {
  'get_popular_movies': get_popular_movies,
  'get_movie_details': get_movie_details,
  'get_movie_credits': get_movie_credits,
  'get_similar_movies': get_similar_movies,
}

In [3]:
_TYPE_MAP = {
  str: "string",
  int: "integer",
  float: "number",
  bool: "boolean",
  list: "array",
  dict: "object",
}

def get_completion_tool_param(func: Callable) -> ChatCompletionToolParam:
  sig = inspect.signature(func)
  hints = get_type_hints(func, include_extras = True)

  properties = {}
  required = []

  for param_name, param in sig.parameters.items():
    hint = hints.get(param_name, str)
    
    if get_origin(hint) is Annotated:
      actual_type, *metadata = get_args(hint)
      description = next((m for m in metadata if isinstance(m, str)), "")
    else:
      actual_type = hint
      description = ""

    json_type = _TYPE_MAP.get(actual_type, "string")
    properties[param_name] = {
      "type": json_type,
      "description": description,
    }

    if param.default is inspect.Parameter.empty:
      required.append(param_name)

  return {
    "type": "function",
    "function": {
      "name": func.__name__,
      "description": (inspect.getdoc(func) or ""),
      "parameters": {
        "type": "object",
        "properties": properties,
        "required": required
      }
    }
  }

TOOLS: list[ChatCompletionToolParam] = [
  get_completion_tool_param(func) for func in FUNCTION_MAP.values()
]

In [4]:


def process_ai_response(message: ChatCompletionMessage):
  
  if not message.tool_calls:
    messages.append({ "role": "assistant", "content": message.content })
    print(f"Agent: {message.content}")
    return

  messages.append({
    "role": "assistant",
    "content": message.content or "",
    "tool_calls": [
      {
        "id": tool_call.id,
        "type": "function",
        "function": {
          "name": tool_call.function.name,
          "arguments": tool_call.function.arguments
        }
      } for tool_call in message.tool_calls
    ]
  })

  for tool_call in message.tool_calls:
    tool_name = tool_call.function.name

    try:
      arguments = json.loads(tool_call.function.arguments)
    except json.JSONDecodeError:
      arguments = {}
    
    call_function = FUNCTION_MAP.get(tool_name)

    result = call_function(**arguments)
    print(f"Agent: [{tool_name}({arguments}) 호출]")

    messages.append({
      "role": "tool",
      "tool_call_id": tool_call.id,
      "tool_name": tool_name,
      "content": json.dumps(result),
    })

  call_ai()

def call_ai():
  response = client.chat.completions.create(
    model = MODEL,
    messages = messages,
    tools = TOOLS,
  )
  
  process_ai_response(response.choices[0].message)

In [ ]:
while True:
  message = input("Send a message to the LLM...")
  if message == "quit" or message == "q":
    break
  else:
    print(f"User: {message}")
    messages.append({ "role": "user", "content": message })
    call_ai()

User: 지금 인기 있는 영화 알려줘
Agent: [get_popular_movies({}) 호출]
Agent: 현재 인기 있는 영화는 다음과 같습니다:

1. **Obsession (ID: 1339713)**  
   - 개봉일: 2026-05-13  
   - 개요: "One Wish Willow"를 깨트리면서 자신의 사랑을 얻으려는 한 희망적인 로맨틱의 이야기가 펼쳐집니다. 하지만 욕망은 어두운 대가를 수반하게 됩니다.  
   - 평점: 7.9  
   ![Obsession](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)

2. **Peddi (ID: 1057265)**  
   - 개봉일: 2026-06-03  
   - 개요: 1980년대 안드라 프라데시의 한 용감한 주민이 스포츠를 통해 지역 사회를 결집시켜 강력한 적에 맞섭니다.  
   - 평점: 6.5  
   ![Peddi](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

3. **Hai Jawani Toh Ishq Hona Hai (ID: 1308553)**  
   - 개봉일: 2026-06-04  
   - 개요: 결혼의 우선순위를 두고 갈등을 겪은 Jass가 해외에서 새로운 로맨스를 경험하지만 충격적인 사실에 직면하게 됩니다.  
   - 평점: 5.5  
   ![Hai Jawani Toh Ishq Hona Hai](https://image.tmdb.org/t/p/w780/vmlJvz6qVzYgei2V74GvnmcuQfW.jpg)

4. **The Unknown Man (ID: 879945)**  
   - 개봉일: 2021-10-16  
   - 개요: 플랜드 작가인 루이가 코트 다쥐르에서 고립된 삶을 살면서 창작의 영감을 찾으려 합니다.  
   - 평점: 8.2  
   ![The Unknown Man](https://image.tmdb.

In [6]:
messages

[{'role': 'user', 'content': '지금 인기 있는 영화 알려줘'},
 {'role': 'assistant',
  'content': '',
  'tool_calls': [{'id': 'call_DFju4bgw7Wsej5kbldjzi6Bq',
    'type': 'function',
    'function': {'name': 'get_popular_movies', 'arguments': '{}'}}]},
 {'role': 'tool',
  'tool_call_id': 'call_DFju4bgw7Wsej5kbldjzi6Bq',
  'tool_name': 'get_popular_movies',
  'content': '[{"adult": false, "backdrop_path": "https://image.tmdb.org/t/p/w1280/4k99kV4R1bbbrsnjR205v91Xbin.jpg", "genre_ids": [27, 53], "id": 1339713, "title": "Obsession", "original_language": "en", "original_title": "Obsession", "overview": "After breaking the mysterious \\"One Wish Willow\\" to win his crush\'s heart, a hopeless romantic finds himself getting exactly what he asked for but soon discovers that some desires come at a dark, sinister price.", "popularity": 881.8242, "poster_path": "https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg", "release_date": "2026-05-13", "softcore": false, "video": false, "vote_average": 7

In [7]:
messages = []